In [1]:
%load_ext sql
%config SqlMagic.displaylimit = 100
%sql sqlite:///PS1.db

Connecting to 'sqlite:///PS1.db'

Problem Set 1 [100 points]
=======

### Deliverables:

Submit your queries (and only those) using the `submission_template.txt` file that is posted on Canvas. Follow the instructions on the file! Upload the file at Canvas.


### Instructions / Notes:

* Run the top cell above to load the database `PS1.db` (make sure the database file, `PS1.db`, is in the same directory as this IPython notebook is running in)
* Some of the problems involve _changing_ this database - you can always re-download `PS1.db` or make a copy if you want to start fresh!
* You **may** create new IPython notebook cells to use for e.g. testing, debugging, exploring, etc.- this is encouraged in fact!- **just make sure that your final answer for each question is _in its own cell_ and _clearly indicated_**
* When you see `In [*]:` to the left of the cell you are executing, this means that the code / query is _running_.
    * **If the cell is hanging- i.e. running for too long: To restart the SQL connection, you must restart the entire python kernel**
    * To restart kernel using the menu bar: "Kernel >> Restart >> Clear all outputs & restart"), then re-execute the sql connection cell at top
    * You will also need to restart the connection if you want to load a different version of the database file
* Remember:
    * `%sql [SQL]` is for _single line_ SQL queries
    * `%%sql [SQL]` is for _multi line_ SQL queries
* _Have fun!_

Problem 1: Linear Algebra [35 points]
------------------------

Two random 3x3 ($N=3$) matrices have been provided in tables `A` and `B`, having the following schema:
> * `i INT`:   Row index
> * `j INT`:   Column index
> * `val INT`: Cell value

**Note: all of your answers below _must_ work for any _square_ matrix sizes, i.e. any value of $N$**.

Note how the matrices are represented - why do we choose this format?  Run the following queries to see the matrices in a nice format:

In [3]:
%sql SELECT group_concat(val, ' , ') AS "A" FROM A GROUP BY i;

Running query in 'sqlite:///PS1.db'

A
"7 , 5 , 8"
"10 , 7 , 7"
"2 , 0 , 5"


In [4]:
%sql SELECT group_concat(val, ' , ') AS "B" FROM B GROUP BY i;

Running query in 'sqlite:///PS1.db'

B
"9 , 6 , 10"
"7 , 6 , 9"
"1 , 1 , 7"


### Part (a): Matrix addition [5 points]

The sum of a matrix $A$ (having dimensions $n\times m$) and a matrix $B$ (having dimensions $n\times m$) is the matrix $C$ (of dimension $n\times m$) having cell at row $i$ and column $j$ equal to:

$C_{ij} = A_{ij} + B_{ij}$

Write a single SQL query to get the sum of $A$ and $B$ (in the same format as $A$ and $B$):

In [6]:
"""
Expected output below- don't re-evaluate this cell!

NOTE: A valid answer must work for ALL inputs of the given type,
not just this example.  I.e. do not hardcode around this answer / etc!
"""

Running query in 'sqlite:///PS1.db'

i,j,val
0,0,16
0,1,11
0,2,18
1,0,17
1,1,13
1,2,16
2,0,3
2,1,1
2,2,12


In [5]:
%sql SELECT A.i, A.j, A.val + B.val AS val FROM A JOIN B ON A.i = B.i AND A.j = B.j;

Running query in 'sqlite:///PS1.db'

i,j,val
0,0,16
0,1,11
0,2,18
1,0,17
1,1,13
1,2,16
2,0,3
2,1,1
2,2,12


### Part (b): Dot product [5 points]

The _dot product_ of two vectors

$a = \begin{bmatrix}a_1 & a_2 & \dots & a_n\end{bmatrix}$

and

$b = \begin{bmatrix}b_1 & b_2 & \dots & b_n\end{bmatrix}$

is

$a\cdot b = \sum_{i=1}^n a_ib_i = a_1b_1 + a_2b_2 + \dots + a_nb_n$

Write a _single SQL query_ to take the dot product of the **first column of $A$** and the **second column of $B$.**:

In [7]:
"""
Expected output below- don't re-evaluate this cell!

NOTE: A valid answer must work for ALL inputs of the given type,
not just this example.  I.e. do not hardcode around this answer / etc!
"""

Running query in 'sqlite:///PS1.db'

DotProduct
104


In [6]:
%sql SELECT SUM(A.val * B.val) AS DotProduct FROM A JOIN B ON A.i = B.i WHERE A.j = 0 and B.j = 1;

Running query in 'sqlite:///PS1.db'

DotProduct
104


### Part (c): Matrix multiplication [10 points]

The product of a matrix $A$ (having dimensions $n\times m$) and a matrix $B$ (having dimensions $m\times p$) is the matrix $C$ (of dimension $n\times p$) having cell at row $i$ and column $j$ equal to:

$C_{ij} = \sum_{k=1}^m A_{ik}B_{kj}$

In other words, to multiply two matrices, get each cell of the resulting matrix $C$, $C_{ij}$, by taking the _dot product_ of the $i$th row of $A$ and the $j$th column of $B$.

Write a single SQL query to get the matrix product of $A$ and $B$ (in the same format as $A$ and $B$):

In [8]:
"""
Expected output below- don't re-evaluate this cell!

NOTE: A valid answer must work for ALL inputs of the given type,
not just this example.  I.e. do not hardcode around this answer / etc!
"""

Running query in 'sqlite:///PS1.db'

i,j,val
0,0,106
0,1,80
0,2,171
1,0,146
1,1,109
1,2,212
2,0,23
2,1,17
2,2,55


In [7]:
%sql SELECT A.i AS i, B.j AS j, SUM(A.val * B.val) AS val FROM A JOIN B ON A.j = B.i GROUP BY A.i, B.j;

Running query in 'sqlite:///PS1.db'

i,j,val
0,0,106
0,1,80
0,2,171
1,0,146
1,1,109
1,2,212
2,0,23
2,1,17
2,2,55


### A note on the `CASE` statement

Sometimes we want a query to return a value that *we* decide, based on a condition. That is what the `CASE` expression does:

```sql
CASE WHEN <condition> THEN <value>
     WHEN <condition> THEN <value>
     ...
     ELSE <value>
END
```

SQLite checks each `WHEN` in order, and the whole expression evaluates to the `<value>` of the first
condition that is true; if none are true, it evaluates to the `ELSE` value. If you omit the `ELSE` and no condition matches, the result is `NULL`.

First, `CASE` in the `SELECT` list, classifying each cell of `A` — much like a computed column:

In [9]:
%%sql
SELECT i, j, val,
       CASE WHEN val = 0 THEN 'zero'
            WHEN val < 6 THEN 'small'
            ELSE 'large' END AS Magnitude
FROM A ORDER BY i, j;

Running query in 'sqlite:///PS1.db'

i,j,val,Magnitude
0,0,7,large
0,1,5,small
0,2,8,large
1,0,10,large
1,1,7,large
1,2,7,large
2,0,2,small
2,1,0,zero
2,2,5,small


`CASE` can also go *inside* an aggregate. Here, it zeroes out every
off-diagonal cell, so the `SUM` adds up only the diagonal — that is, the **trace** of `A`:

In [10]:
%%sql
SELECT SUM(CASE WHEN i = j THEN val ELSE 0 END) AS Trace FROM A;

Running query in 'sqlite:///PS1.db'

Trace
19


Finally — and this is the pattern you will need below — a condition may itself be a subquery. The
`EXISTS` operator is true when its subquery returns at least one row, and `NOT EXISTS` is true when
it returns none. Combining the two lets a query report a yes/no answer as a single row:

In [11]:
%%sql
SELECT CASE WHEN EXISTS (SELECT 1 FROM A WHERE val = 0)
            THEN 'TRUE' ELSE 'FALSE' END AS HasAZero;

Running query in 'sqlite:///PS1.db'

HasAZero
TRUE


### Part (d): Checking Matrix Symmetry [7 points]

A matrix is **symmetric** when $A_{ij} = A_{ji}$ for every $i$ and every $j$.

Write a _single SQL query_ that returns **exactly one row with one column**, named `IsSymmetric`,
whose value is the string `'TRUE'` if $A$ is symmetric and `'FALSE'` otherwise.

Write your query here:

In [12]:
"""
Expected output below- don't re-evaluate this cell!

NOTE: A valid answer must work for ALL inputs of the given type,
not just this example.  I.e. do not hardcode around this answer / etc!
"""

Running query in 'sqlite:///PS1.db'

IsSymmetric
FALSE


In [8]:
%%sql
SELECT CASE WHEN EXISTS (SELECT 1 FROM A AS x JOIN A AS y ON x.i = y.j AND x.j = y.i WHERE x.val != y.val)
            THEN 'FALSE' ELSE 'TRUE' END AS IsSymmetric;

Running query in 'sqlite:///PS1.db'

IsSymmetric
FALSE


### Part (e): Checking Commutativity [8 points]

Matrix multiplication is in general **not commutative**: for most matrices, $A \cdot B \neq B \cdot A$. 

Write a _single SQL query_ that returns **exactly one row with one column**, named
`IsCommutative`, whose value is the string `'TRUE'` if $A \cdot B = B \cdot A$ and `'FALSE'`
otherwise.

*Hint:* SQL makes it easy to say "**there exists** a cell where the two matrices differ," and hard
to say "**for every** cell the two matrices agree." But these two statements are negations of one
another: the second is true exactly when the first is false.


In [13]:
"""
Expected output below- don't re-evaluate this cell!

NOTE: A valid answer must work for ALL inputs of the given type,
not just this example.  I.e. do not hardcode around this answer / etc!
"""

Running query in 'sqlite:///PS1.db'

IsCommutative
FALSE


In [9]:
%%sql
WITH AB AS (SELECT A.i AS i, B.j AS j, SUM(A.val * B.val) AS val FROM A JOIN B ON A.j = B.i GROUP BY A.i, B.j),
BA AS (SELECT B.i AS i, A.j AS j, SUM(B.val * A.val) AS val FROM B JOIN A ON B.j = A.i GROUP BY B.i, A.j)
SELECT CASE WHEN EXISTS (SELECT 1 FROM AB JOIN BA ON AB.i = BA.i AND AB.j = BA.j WHERE AB.val != BA.val)
            THEN 'FALSE' ELSE 'TRUE' END AS IsCommutative;

Running query in 'sqlite:///PS1.db'

IsCommutative
FALSE


Problem 2: The Sales Database [30 points]
----------------------------------------------

We've prepared and loaded a dataset related to sales data from a company. The dataset has the following schema:

> `Holidays (WeekDate, IsHoliday)`

> `Stores (Store, Type, Size)`

> `TemporalData(Store, WeekDate, Temperature, FuelPrice, CPI, UnemploymentRate)`

> `Sales (Store, Dept, WeekDate, WeeklySales)`

Before you start writing queries on the database, find the schema and the constraints (keys, foreign keys). 

### Part (a): Sales during Holidays [8 points]

Using a _single SQL query_, find the store(s) with the largest overall sales during holiday weeks. Further requirements:
* Use the `WITH` clause before the main body of the query to compute a subquery if necessary.
* Return a relation with schema `(Store, AllSales)`.
* Round `AllSales` to two decimal places with `ROUND(<value>, 2)`.

Write your query here:

In [14]:
"""
Expected output below- don't re-evaluate this cell!

NOTE: A valid answer must work for ALL inputs of the given type,
not just this example.  I.e. do not hardcode around this answer / etc!
"""

Running query in 'sqlite:///PS1.db'

Store,AllSales
20,22490350.81


In [10]:
%%sql
WITH HolidaySales AS (SELECT s.Store AS Store, SUM(s.WeeklySales) AS TotalSales
                      FROM Sales AS s
                      JOIN Holidays AS h ON s.WeekDate = h.Weekdate
                      WHERE h.IsHoliday = 'TRUE'
                      GROUP BY s.Store)
SELECT Store, ROUND(TotalSales, 2) AS AllSales
FROM HolidaySales
WHERE TotalSales = (SELECT MAX(TotalSales) FROM HolidaySales);

Running query in 'sqlite:///PS1.db'

Store,AllSales
20,22490350.81


### Part (b): When Holidays do not help Sales [12 points]

Using a _single SQL query_, compute the **number** of non-holiday weeks that had larger sales than the overall average sales during holiday weeks. Further requirements:
* Use the `WITH` clause before the main body of the query to compute a subquery if necessary.
* Return a relation with schema `(NumberNonHolidays)`.

Write your query here:

In [15]:
"""
Expected output below- don't re-evaluate this cell!

NOTE: A valid answer must work for ALL inputs of the given type,
not just this example.  I.e. do not hardcode around this answer / etc!
"""

Running query in 'sqlite:///PS1.db'

NumberNonHolidays
8


In [11]:
%%sql
WITH WeeklySales AS (SELECT s.WeekDate, SUM(s.WeeklySales) AS TotalSales 
                    FROM Sales AS s GROUP BY s.WeekDate),
HolidayAverage AS (SELECT AVG(w.TotalSales) AS AvgHolidaySales 
                    FROM WeeklySales AS w JOIN Holidays AS h ON h.WeekDate = w.WeekDate
                    WHERE h.IsHoliday = 'TRUE')
SELECT COUNT(*) AS NumberNonHolidays
FROM WeeklySales AS w 
JOIN HolidayAverage AS h ON 1=1
JOIN Holidays AS d ON w.WeekDate = d.WeekDate
WHERE w.TotalSales > h.AvgHolidaySales and d.IsHoliday = 'FALSE';

Running query in 'sqlite:///PS1.db'

NumberNonHolidays
8


### Part (c): Sales by season [10 points]

Using a _single SQL query_, compute the **average weekly sales in each season, for each type of
store** — that is, a table with one row per store type and one column per season. Further
requirements:

* Return a relation with schema `(Type, Winter, Spring, Summer, Fall)`.
* The seasons are Winter = months 12, 1, 2; Spring = 3, 4, 5; Summer = 6, 7, 8; Fall = 9, 10, 11.
* Round every average to two decimal places with `ROUND(<value>, 2)`.

*Hint:* SQLite3 does not support native operations on the DATE datatype. To create a workaround, you
can use the `LIKE` predicate and the string concatenation operator (`||`). You can also use the
substring operator that SQLite3 supports (`substr`).

*Hint:* Reshaping rows into columns like this is called a **pivot**, and you can use `CASE` to implement it.

Write your query here:

In [16]:
"""
Expected output below- don't re-evaluate this cell!

NOTE: A valid answer must work for ALL inputs of the given type,
not just this example.  I.e. do not hardcode around this answer / etc!
"""

Running query in 'sqlite:///PS1.db'

Type,Winter,Spring,Summer,Fall
A,20884.92,19653.21,20205.38,19792.35
B,12858.74,11869.71,12378.46,11941.35
C,9583.17,9535.29,9415.71,9568.26


In [12]:
%%sql
WITH SeasonWeeklySales AS (SELECT t.Type, s.WeeklySales, substr(s.WeekDate, 6, 2) AS Month
                        FROM Sales AS s JOIN Stores AS t ON s.Store = t.Store)
SELECT Type, 
ROUND(AVG(CASE WHEN Month IN ('12', '01', '02') THEN WeeklySales END), 2) AS Winter,
ROUND(AVG(CASE WHEN Month IN ('03', '04', '05') THEN WeeklySales END), 2) AS Spring,
ROUND(AVG(CASE WHEN Month IN ('06', '07', '08') THEN WeeklySales END), 2) AS Summer,
ROUND(AVG(CASE WHEN Month IN ('09', '10', '11') THEN WeeklySales END), 2) AS Fall
FROM SeasonWeeklySales
GROUP BY Type;

Running query in 'sqlite:///PS1.db'

Type,Winter,Spring,Summer,Fall
A,20884.92,19653.21,20205.38,19792.35
B,12858.74,11869.71,12378.46,11941.35
C,9583.17,9535.29,9415.71,9568.26


Problem 3: The Traveling SQL Server Salesman Problem [35 points]
--------------------------------------------------

SQL Server salespeople are lucky as far as traveling salespeople go- they only have to sell one or two big enterprise contracts, at one or two offices in Wisconsin, in order to make their monthly quota!

Answer the following questions using the table of streets connecting company office buildings.

**Note that for convenience all streets are included _twice_, as $A \rightarrow B$ and $B \rightarrow A$.  This should make some parts of the problem easier, but remember to take it into account!**

In [17]:
%sql SELECT * FROM streets LIMIT 4;

Running query in 'sqlite:///PS1.db'

id,direction,A,B,d
0,F,UW-Madison,DooHickey Collective,7
0,R,DooHickey Collective,UW-Madison,7
1,F,DooHickey Collective,Gizmo Corp,2
1,R,Gizmo Corp,DooHickey Collective,2


### Part (a): One-hop, two-hop, three-hop... [10 points]

Our salesperson has stopped at UW-Madison, to steal some cool new RDBMS technology from CS564-ers, and now wants to go sell it to a company _within 10 miles of UW-Madison_ and _passing through no more than 3 distinct streets_.  Write a single query, not using `WITH` (see later on), to find all such companies.

Your query should return the schema `(company, distance)` where distance is cumulative from UW-Madison.

Write your query here:

In [18]:
"""
Expected output below- don't re-evaluate this cell!

NOTE: A valid answer must work for ALL inputs of the given type,
not just this example.  I.e. do not hardcode around this answer / etc!
"""

Running query in 'sqlite:///PS1.db'

company,distance
DooHickey Collective,7
DooHickey Corp,9
Gadget Collective,9
Gadget Corp,6
Gizmo Corp,9
Widget Industries,10


### Part (b): Cul-de-sacs [5 points]

Some of these offices are awkward to visit: exactly **one** street connects them to the rest of the
map, so the only way out is back the way you came. Our salesperson would like to know which
companies those are, and (since there is only one choice) which company each of them leads to.

Write a single query, **not** using `WITH`, that returns every company served by exactly one street,
together with the company at the other end of that street.

Your query should return the schema `(Company, ConnectsTo)`, ordered by `Company`.

*Remember:* every street is stored **twice**, once as $A \rightarrow B$ and once as
$B \rightarrow A$. Take that into account when you count the streets that serve a company.

Write your query here:

In [19]:
"""
Expected output below- don't re-evaluate this cell!

NOTE: A valid answer must work for ALL inputs of the given type,
not just this example.  I.e. do not hardcode around this answer / etc!
"""

Running query in 'sqlite:///PS1.db'

Company,ConnectsTo
GadgetWorks,Gizmo Corp
Gizmo Collective,Thing Industries
Gizmo Industries,Widget Industries
GizmoWorks,Gadget Collective
Thing Collective,DooHickeyCo
ThingCo,DooHickey Corp
ThingWorks,GizmoCo
Widget Corp,Gadget Industries
WidgetCo,DooHickey Industries
WidgetWorks,Widget Collective


### Part (c): A stop at the Farm [10 points]

Now, our salesperson is out in the field, and wants to see all routes- and their distances- which will take him/her from a company $A$ to a company $B$, with the following constraints:
* The route must pass through UW-Madison (in order to pick up new RDBMS tech to sell!)
* $A$ and $B$ must _each individually_ be within 2 hops of UW-Madison
* $A$ and $B$ must be different companies
* _The total distance must be $<= 15$_
* Do not use `WITH`
* If you return a path $A \rightarrow B$, _do not include_ $B \rightarrow A$ in your answer!

In order to make your answer a bit cleaner, you may split into two queries, one of which creates a `VIEW`.  A view is a virtual table based on the output set of a SQL query.  A view can be used just like a normal table- the only difference under the hood is that the DBMS re-evaluates the query used to generate it each time a view is queried by a user (thus the data is always up-to date!)

Here's a simple example of a view:

In [20]:
%%sql 
DROP VIEW IF EXISTS short_streets;
CREATE VIEW short_streets AS 
SELECT A, B, d FROM streets WHERE d < 3;
SELECT * FROM short_streets LIMIT 3;

Running query in 'sqlite:///PS1.db'

++
||
++
++

Write your query or queries here:

In [21]:
"""
Expected output below- don't re-evaluate this cell!

NOTE: A valid answer must work for ALL inputs of the given type,
not just this example.  I.e. do not hardcode around this answer / etc!
"""

Running query in 'sqlite:///PS1.db'

company_1,company_2,distance
Gadget Corp,DooHickey Collective,13
Gadget Corp,DooHickey Corp,15
Gadget Corp,Gadget Collective,15
Gizmo Corp,Gadget Corp,15


### Part (d): Finding Triangles [10 points]

Finally, our salesperson wants to find the longest route that goes from some company $A$ to company $B$ to company $C$ and then back to company $A$. Also, the following constraints must hold:
* $A$, $B$, $C$ must be different companies
* Do not use `WITH` 
* Output only the length of the longest route

Write your query here:

In [22]:
"""
Expected output below- don't re-evaluate this cell!

NOTE: A valid answer must work for ALL inputs of the given type,
not just this example.  I.e. do not hardcode around this answer / etc!
"""

Running query in 'sqlite:///PS1.db'

LongestDistance
18
